# Set Up

In [1]:
import pandas as pd
import numpy as np
from pcj.utils import dados, BASE_DIR
from pcj.processamento import decompor_estatisticas
from pcj.modelos_ML import isolation_forest
from pcj.outliers import (detect_outliers_zscore, detect_outliers_iqr, detect_outliers_pca_kmeans,
                          detect_outliers_lof, detect_outliers_knn, detect_outliers_ensemble,
                          detect_outliers_estatisticos, comparar_marcacoes, resumo_por_mes, tratar_outliers)

In [2]:
df, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")
df_med = decompor_estatisticas(df, estatistica="med")


Verificação do efeito da transformação logarítmica

In [3]:
df_med[variaveis].skew()

COR              3.431387
TURB.            1.982049
pH               0.640509
ALC.             1.800605
AC.              3.818553
O.C.             2.850822
O.D.             0.181960
Cl               3.915182
DUR.             1.407600
Fe               2.665936
Mn               2.408677
Cond.            1.524526
Cianobacteria    3.639978
C.F.             3.765265
Clorofila        6.107537
F                0.629216
dtype: float64

In [4]:
np.log1p(df_med[variaveis]).skew()

COR              0.498347
TURB.            0.256828
pH               0.569107
ALC.             0.386058
AC.              1.523625
O.C.             0.759782
O.D.            -0.693960
Cl               0.854930
DUR.            -0.111096
Fe               0.946183
Mn               2.037038
Cond.            0.145859
Cianobacteria   -0.258637
C.F.            -0.618300
Clorofila       -0.081824
F                0.114069
dtype: float64

# Z-Score

In [5]:
limites_z, total_z, mascara_z = detect_outliers_zscore(df_med, variaveis, verbose=True)
limites_z


Total de outliers (Z-Score, somando todas as colunas): 52
Total de linhas com pelo menos um outlier (Z-Score): 30


,limite_inferior,limite_superior,n_outliers
COR,-209.678710,450.631590,3
TURB.,-165.188988,292.340820,3
pH,6.784348,7.864867,3
ALC.,-16.999291,143.845444,5
AC.,-15.508805,39.981332,4
O.C.,-0.125862,17.205862,3
O.D.,0.469263,5.769240,0
Cl,-35.333001,124.716785,3
DUR.,1.234158,106.711788,2
Fe,-3.701493,8.009709,4


In [6]:
limites_z_log, total_z_log, mascara_z_log = detect_outliers_zscore(df_med, variaveis, log=True, verbose=True)
limites_z_log


Total de outliers (Z-Score, somando todas as colunas): 32
Total de linhas com pelo menos um outlier (Z-Score): 22


,limite_inferior,limite_superior,n_outliers
COR,10.455819,7.529917e+02,2
TURB.,0.099675,1.060523e+03,0
pH,6.802833,7.877152e+00,3
ALC.,18.722207,1.811609e+02,1
AC.,2.044521,4.354708e+01,4
O.C.,3.259408,1.889550e+01,3
O.D.,1.040214,6.921207e+00,2
Cl,10.601946,1.453528e+02,2
DUR.,19.893110,1.306497e+02,2
Fe,-0.335685,1.053610e+01,1


In [7]:
_, total_zscore_log, mascara_zscore_log = detect_outliers_zscore(df_med, variaveis, log=True)
resumo_por_mes(df_med, mascara_zscore_log)


,Data,n_variaveis,variaveis
202,2014-08-01,4,"ALC., AC., O.C., O.D."
34,2009-12-01,2,"COR, Fe"
184,2014-02-01,2,"pH, Clorofila"
211,2014-11-01,2,"Cl, F"
208,2014-10-01,2,"pH, Cl"
205,2014-09-01,2,"pH, O.C."
319,2017-11-01,2,"AC., Mn"
286,2016-12-01,2,"Cianobacteria, Clorofila"
4,2009-02-01,1,COR
61,2010-09-01,1,O.D.


# IQR

In [8]:
limites_iqr, total_iqr, mascara_iqr = detect_outliers_iqr(df_med, variaveis, verbose=True)
limites_iqr



Total de outliers (IQR, somando todas as colunas): 181
Total de linhas com pelo menos um outlier (IQR): 89


,limite_inferior,limite_superior,n_outliers
COR,-93.250,300.750,7
TURB.,-108.500,207.500,14
pH,6.900,7.700,4
ALC.,5.500,113.500,11
AC.,2.000,18.000,21
O.C.,2.750,13.550,5
O.D.,0.775,5.375,4
Cl,3.500,79.500,13
DUR.,10.500,94.500,6
Fe,-1.810,5.470,11


In [9]:
resumo_por_mes(df_med, mascara_iqr)


,Data,n_variaveis,variaveis
202,2014-08-01,9,"ALC., AC., O.C., O.D., Cl, DUR., Cond., C.F., F"
205,2014-09-01,8,"pH, ALC., O.C., Cl, DUR., Mn, Cond., F"
208,2014-10-01,6,"pH, ALC., Cl, DUR., Cond., C.F."
211,2014-11-01,6,"ALC., Cl, Mn, Cond., C.F., F"
199,2014-07-01,5,"ALC., AC., Cl, Cond., C.F."
...,...,...,...
511,2023-03-01,1,AC.
517,2023-05-01,1,DUR.
514,2023-04-01,1,AC.
556,2024-06-01,1,AC.


In [10]:
limites_iqr_log, total_iqr_log, mascara_iqr_log = detect_outliers_iqr(df_med, variaveis, log=True, verbose=True)
limites_iqr_log



Total de outliers (IQR, somando todas as colunas): 81
Total de linhas com pelo menos um outlier (IQR): 52


,limite_inferior,limite_superior,n_outliers
COR,11.007767,710.700376,2
TURB.,-0.529281,2100.086118,0
pH,6.908893,7.709183,4
ALC.,22.790144,145.194998,4
AC.,4.184313,21.568080,18
O.C.,3.994046,15.399528,4
O.D.,1.285748,6.119796,2
Cl,15.683218,101.857852,7
DUR.,22.681062,115.211004,3
Fe,-0.293771,9.167804,3


In [11]:
resumo_por_mes(df_med, mascara_iqr_log)

,Data,n_variaveis,variaveis
202,2014-08-01,6,"ALC., AC., O.C., O.D., Cl, F"
205,2014-09-01,6,"pH, ALC., O.C., Cl, Mn, F"
208,2014-10-01,4,"pH, ALC., Cl, Cond."
199,2014-07-01,3,"ALC., AC., Cl"
211,2014-11-01,3,"Cl, Mn, F"
301,2017-05-01,3,"AC., C.F., F"
508,2023-02-01,3,"AC., Cl, Mn"
184,2014-02-01,2,"pH, Clorofila"
286,2016-12-01,2,"Cianobacteria, Clorofila"
256,2016-02-01,2,"Cl, DUR."


# K-Means
PCA + K-Means em grade: 6 contaminações × 30 seeds = 180 rodadas. Só avalia os meses completos (sem NaN).

In [12]:
freq_km, cons_km = detect_outliers_pca_kmeans(df_med, variaveis, verbose=True)
df_med.loc[freq_km > 0, ["Data"]].assign(frequencia=freq_km, consenso=cons_km).sort_values("frequencia", ascending=False)

PCA + K-Means aplicado em 126 meses completos (65 com NaN, não avaliados); 12 componentes principais
6 contaminações x 30 seeds = 180 rodadas; meses marcados em pelo menos 50% das rodadas: 10


,Data,frequencia,consenso
184,2014-02-01,1.000000,True
34,2009-12-01,1.000000,True
466,2021-12-01,0.994444,True
256,2016-02-01,0.983333,True
199,2014-07-01,0.944444,True
211,2014-11-01,0.933333,True
247,2015-11-01,0.861111,True
205,2014-09-01,0.861111,True
496,2022-10-01,0.711111,True
190,2014-04-01,0.527778,True


In [13]:
print(cons_km.dtype)                       # esperado: bool
print(cons_km.index.equals(df_med.index))  # esperado: True
print(freq_km.notna().sum())               # quantos meses foram avaliados?

bool
True
126


# Isolation Forest
Grade: 6 contaminações (0,05 a 0,10) × 30 seeds = 180 rodadas. `frequencia` = fração das rodadas em que o mês foi marcado; `consenso` = frequência ≥ 0,5.

In [14]:
freq_if, cons_if = isolation_forest(df_med, variaveis, verbose=True)
df_med.loc[freq_if > 0, ["Data"]].assign(frequencia=freq_if, consenso=cons_if).sort_values("frequencia", ascending=False)

6 contaminações x 30 seeds = 180 rodadas
Linhas marcadas em pelo menos 50% das rodadas: 13


,Data,frequencia,consenso
34,2009-12-01,1.000000,True
205,2014-09-01,1.000000,True
202,2014-08-01,1.000000,True
211,2014-11-01,1.000000,True
208,2014-10-01,1.000000,True
199,2014-07-01,1.000000,True
256,2016-02-01,1.000000,True
184,2014-02-01,0.994444,True
283,2016-11-01,0.927778,True
4,2009-02-01,0.927778,True


# LOF
Grade: as mesmas 6 contaminações, sem seeds (o LOF não tem aleatoriedade). Só avalia os meses completos (sem NaN).

In [15]:
freq_lof, cons_lof = detect_outliers_lof(df_med, variaveis, verbose=True)
df_med.loc[freq_lof > 0, ["Data"]].assign(frequencia=freq_lof, consenso=cons_lof).sort_values("frequencia", ascending=False)

LOF aplicado em 126 meses completos (65 com NaN, não avaliados)
6 contaminações; meses marcados em pelo menos 50% das rodadas: 10


,Data,frequencia,consenso
184,2014-02-01,1.000000,True
247,2015-11-01,1.000000,True
211,2014-11-01,1.000000,True
205,2014-09-01,1.000000,True
199,2014-07-01,1.000000,True
256,2016-02-01,1.000000,True
466,2021-12-01,1.000000,True
34,2009-12-01,0.833333,True
496,2022-10-01,0.666667,True
190,2014-04-01,0.500000,True


# KNN
Score = distância ao 5º vizinho mais próximo. Mesma grade e mesmos meses avaliados do LOF.

In [16]:
freq_knn, cons_knn = detect_outliers_knn(df_med, variaveis, verbose=True)
df_med.loc[freq_knn > 0, ["Data"]].assign(frequencia=freq_knn, consenso=cons_knn).sort_values("frequencia", ascending=False)

KNN aplicado em 126 meses completos (65 com NaN, não avaliados)
6 contaminações; meses marcados em pelo menos 50% das rodadas: 10


,Data,frequencia,consenso
34,2009-12-01,1.000000,True
199,2014-07-01,1.000000,True
184,2014-02-01,1.000000,True
211,2014-11-01,1.000000,True
205,2014-09-01,1.000000,True
247,2015-11-01,1.000000,True
256,2016-02-01,1.000000,True
466,2021-12-01,0.833333,True
496,2022-10-01,0.666667,True
190,2014-04-01,0.500000,True


## LOF × KNN
Os dois marcam os mesmos meses? (linhas: LOF; colunas: KNN)

In [17]:
pd.crosstab(cons_lof, cons_knn, rownames=["LOF"], colnames=["KNN"])

KNN,False,True
LOF,,
False,181,0
True,0,10


# Ensemble (IF + LOF + KNN)
Padrão: voto por maioria entre os detectores que avaliaram o mês. Nos meses com NaN, só o IF avalia.

In [18]:
tabela_ens = detect_outliers_ensemble(df_med, variaveis, verbose=True)
tabela_ens[tabela_ens["votos"] > 0]

Ensemble (IF + LOF + KNN, regra 'maioria'): 14 meses marcados
Meses avaliados por um único detector: 65


,Data,IF,LOF,KNN,avaliaram,votos,media,ensemble
4,2009-02-01,True,False,False,3,1,0.531481,False
34,2009-12-01,True,True,True,3,3,0.944444,True
73,2011-01-01,True,False,False,3,1,0.305556,False
184,2014-02-01,True,True,True,3,3,0.998148,True
190,2014-04-01,False,True,True,3,2,0.333333,True
199,2014-07-01,True,True,True,3,3,1.000000,True
202,2014-08-01,True,False,False,1,1,1.000000,True
205,2014-09-01,True,True,True,3,3,1.000000,True
208,2014-10-01,True,False,False,1,1,1.000000,True
211,2014-11-01,True,True,True,3,3,1.000000,True


Cada detector × ensemble. Os detectores votaram no ensemble: a tabela mostra a contribuição de cada um, não uma comparação independente.

In [19]:
comparar_marcacoes(tabela_ens, referencia="ensemble")

,marcados,em_comum,so_no_metodo,so_na_referencia,jaccard
IF,13,11,2,3,0.687500
LOF,10,10,0,4,0.714286
KNN,10,10,0,4,0.714286


## Cenários do ensemble
Meses marcados em cada combinação de detectores × regra. Com o K-Means (4 detectores), a maioria exige 3 votos: o empate de 2 a 2 não marca. Cada cenário roda os detectores de novo (a célula leva alguns minutos).

In [20]:
# Chave: (detectores, regra). Valor: marcação do ensemble (True/False para cada mês)
cenarios = {}

for metodos in [("IF", "LOF", "KNN"), ("IF", "LOF"), ("IF", "LOF", "KNN", "K-Means")]:
    for regra in ["maioria", "unanimidade", "qualquer", "media"]:
        tabela = detect_outliers_ensemble(df_med, variaveis, metodos=metodos, regra=regra)
        cenarios[(" + ".join(metodos), regra)] = tabela["ensemble"]

# Tabela: uma linha por mês, uma coluna por cenário
cenarios = pd.DataFrame(cenarios)

# Meses marcados em cada cenário (linhas: detectores; colunas: regra)
cenarios.sum().unstack()[["maioria", "unanimidade", "qualquer", "media"]]

,maioria,unanimidade,qualquer,media
IF + LOF,11,11,16,13
IF + LOF + KNN,14,11,16,14
IF + LOF + KNN + K-Means,14,11,16,13


Em quantos dos 12 cenários cada mês foi marcado (12 = marcado em todos).

In [21]:
n_cenarios = cenarios.sum(axis=1)
df_med.loc[n_cenarios > 0, ["Data"]].assign(n_cenarios=n_cenarios).sort_values("n_cenarios", ascending=False)

,Data,n_cenarios
34,2009-12-01,12
184,2014-02-01,12
202,2014-08-01,12
199,2014-07-01,12
205,2014-09-01,12
247,2015-11-01,12
211,2014-11-01,12
208,2014-10-01,12
268,2016-06-01,12
283,2016-11-01,12


Cada cenário × ensemble padrão (IF + LOF + KNN, maioria).

In [22]:
comparar_marcacoes(cenarios, referencia=("IF + LOF + KNN", "maioria"))

marcados  em_comum  so_no_metodo  \
IF + LOF + KNN           unanimidade        11        11             0   
                         qualquer           16        14             2   
                         media              14        13             1   
IF + LOF                 maioria            11        11             0   
                         unanimidade        11        11             0   
                         qualquer           16        14             2   
                         media              13        12             1   
IF + LOF + KNN + K-Means maioria            14        14             0   
                         unanimidade        11        11             0   
                         qualquer           16        14             2   
                         media              13        13             0   

                                      so_na_referencia   jaccard  
IF + LOF + KNN           unanimidade                 3  0.785714  
                         qualquer                    0  0.875000  
                         media                       1  0.866667  
IF + LOF                 maioria                     3  0.785714  
                         unanimidade                 3  0.785714  
                         qualquer                    0  0.875000  
                         media                       2  0.800000  
IF + LOF + KNN + K-Means maioria                     0  1.000000  
                         unanimidade                 3  0.785714  
                         qualquer                    0  0.875000  
                         media                       1  0.928571

# Comparação: ensemble × Z-Score, IQR e K-Means
Ensemble padrão (IF + LOF + KNN, maioria): sem o K-Means, para a comparação com ele não ser circular. Z-Score e IQR em escala log; o mês conta quando pelo menos 1 variável foi marcada.

In [23]:
# Uma linha por mês: Z-Score e IQR, mais as marcações do ensemble e do K-Means
comparacao = detect_outliers_estatisticos(df_med, variaveis, log=True, min_variaveis=1, verbose=True)
comparacao["ensemble"] = tabela_ens["ensemble"]
comparacao["K-Means"] = cons_km

# Quantos dos quatro métodos marcaram cada mês (True vale 1)
comparacao["n_metodos"] = comparacao[["ensemble", "Z-Score", "IQR", "K-Means"]].sum(axis=1)

comparar_marcacoes(comparacao, referencia="ensemble", metodos=["Z-Score", "IQR", "K-Means"])

Z-Score e IQR em escala log; mês marcado com pelo menos 1 variável(is)
Meses marcados: Z-Score 22, IQR 52


,marcados,em_comum,so_no_metodo,so_na_referencia,jaccard
Z-Score,22,11,11,3,0.440000
IQR,52,12,40,2,0.222222
K-Means,10,10,0,4,0.714286


In [24]:
comparacao[comparacao["n_metodos"] > 0].sort_values(["n_metodos", "Data"], ascending=[False, True])

,Data,n_var_avaliadas,n_var_zscore,n_var_iqr,Z-Score,IQR,ensemble,K-Means,n_metodos
34,2009-12-01,16,2,2,True,True,True,True,4
184,2014-02-01,16,2,2,True,True,True,True,4
199,2014-07-01,16,1,3,True,True,True,True,4
205,2014-09-01,16,2,6,True,True,True,True,4
211,2014-11-01,16,2,3,True,True,True,True,4
247,2015-11-01,16,1,1,True,True,True,True,4
256,2016-02-01,16,1,2,True,True,True,True,4
496,2022-10-01,16,1,1,True,True,True,True,4
202,2014-08-01,14,4,6,True,True,True,False,3
208,2014-10-01,15,2,4,True,True,True,False,3


## Meses marcados por ano

In [25]:
comparacao.groupby(comparacao["Data"].dt.year)[["ensemble", "Z-Score", "IQR", "K-Means"]].sum()

,ensemble,Z-Score,IQR,K-Means
Data,,,,
2009,1,2,2,1
2010,0,1,1,0
2011,0,0,1,0
2012,0,0,1,0
2013,0,0,0,0
2014,7,6,7,5
2015,1,1,2,1
2016,3,3,5,1
2017,0,5,8,0


# Tabela-resumo
Meses marcados por método. Percentual sobre todos os meses; `avaliados` = meses que o método consegue avaliar.

In [26]:
# O Z-Score e o IQR avaliam valor a valor: basta o mês ter pelo menos uma variável sem NaN
meses_com_algum_valor = df_med[variaveis].notna().any(axis=1).sum()

# Uma linha por método: meses marcados e meses avaliados
tabela_resumo = {
    "Z-Score (log, ≥ 1 variável)": {"meses": comparacao["Z-Score"].sum(), "avaliados": meses_com_algum_valor},
    "IQR (log, ≥ 1 variável)": {"meses": comparacao["IQR"].sum(), "avaliados": meses_com_algum_valor},
    "PCA + K-Means": {"meses": cons_km.sum(), "avaliados": freq_km.notna().sum()},
    "Isolation Forest": {"meses": cons_if.sum(), "avaliados": freq_if.notna().sum()},
    "LOF": {"meses": cons_lof.sum(), "avaliados": freq_lof.notna().sum()},
    "KNN": {"meses": cons_knn.sum(), "avaliados": freq_knn.notna().sum()},
    "Ensemble (maioria)": {"meses": tabela_ens["ensemble"].sum(), "avaliados": (tabela_ens["avaliaram"] > 0).sum()},
}
tabela_resumo = pd.DataFrame.from_dict(tabela_resumo, orient="index")
tabela_resumo[f"% de {len(df_med)}"] = (100 * tabela_resumo["meses"] / len(df_med)).round(1)
tabela_resumo

,meses,avaliados,% de 191
"Z-Score (log, ≥ 1 variável)",22,191,11.5
"IQR (log, ≥ 1 variável)",52,191,27.2
PCA + K-Means,10,126,5.2
Isolation Forest,13,191,6.8
LOF,10,126,5.2
KNN,10,126,5.2
Ensemble (maioria),14,191,7.3


# Tratamento dos outliers
Um valor vira NaN quando o IQR (escala log) o marca **e** o ensemble padrão marca o mês. O mês continua na base, e a imputação preenche os valores apagados depois. Os meses do ensemble sem nenhum valor extremo (outliers só pela combinação das variáveis) ficam intactos.

In [27]:
# mascara_iqr_log vem da seção IQR; tabela_ens, da seção Ensemble
df_tratado, registro = tratar_outliers(df_med, tabela_ens["ensemble"], mascara_iqr_log, verbose=True)
registro

Valores trocados por NaN: 32, em 12 meses
Meses marcados sem nenhum valor marcado (ficam intactos): 2


,Data,variavel,valor_original
0,2009-12-01,COR,798.00
1,2009-12-01,Fe,14.74
2,2014-02-01,pH,7.90
3,2014-02-01,Clorofila,509.64
4,2014-07-01,ALC.,166.00
5,2014-07-01,AC.,59.00
6,2014-07-01,Cl,131.00
7,2014-08-01,ALC.,195.00
8,2014-08-01,AC.,61.00
9,2014-08-01,O.C.,27.90


Valores trocados por NaN, por variável

In [28]:
registro["variavel"].value_counts()

variavel
Cl           6
Mn           5
ALC.         4
pH           3
F            3
AC.          3
O.C.         2
COR          1
Fe           1
Clorofila    1
O.D.         1
Cond.        1
DUR.         1
Name: count, dtype: int64

Meses completos (sem NaN) antes e depois. O experimento A só usa os meses completos.

In [29]:
print("Meses completos antes do tratamento:", df_med[variaveis].notna().all(axis=1).sum())
print("Meses completos depois do tratamento:", df_tratado[variaveis].notna().all(axis=1).sum())

Meses completos antes do tratamento: 126
Meses completos depois do tratamento: 118


Gravação da base tratada, que os notebooks de imputação e de modelos vão ler. A coluna `Index` (número da linha na base limpa) já faz parte de `df_med`.

In [30]:
df_tratado.to_excel(BASE_DIR / "data" / "processed" / "dados_med_tratados.xlsx", index=False)

# Apêndice
Sensibilidade da comparação do Z-Score e do IQR com o ensemble: escala (log × original) e regra do mês (≥ 1 × ≥ 2 variáveis). A primeira combinação repete a seção Comparação.

In [31]:
# Chave: (escala, regra do mês). Valor: comparação com o ensemble
resumos = {}

for log in [True, False]:
    for min_variaveis in [1, 2]:
        tabela = detect_outliers_estatisticos(df_med, variaveis, log=log, min_variaveis=min_variaveis)
        tabela["ensemble"] = tabela_ens["ensemble"]
        escala = "log" if log else "original"
        resumos[(escala, f"≥ {min_variaveis} variável(is)")] = comparar_marcacoes(tabela, metodos=["Z-Score", "IQR"])

pd.concat(resumos, names=["escala", "regra do mês", "método"])

marcados  em_comum  so_no_metodo  \
escala   regra do mês     método                                      
log      ≥ 1 variável(is) Z-Score        22        11            11   
                          IQR            52        12            40   
         ≥ 2 variável(is) Z-Score         8         6             2   
                          IQR            15         8             7   
original ≥ 1 variável(is) Z-Score        30        14            16   
                          IQR            89        14            75   
         ≥ 2 variável(is) Z-Score        11         9             2   
                          IQR            43        13            30   

                                   so_na_referencia   jaccard  
escala   regra do mês     método                               
log      ≥ 1 variável(is) Z-Score                 3  0.440000  
                          IQR                     2  0.222222  
         ≥ 2 variável(is) Z-Score                 8  0.375000  
                          IQR                     6  0.380952  
original ≥ 1 variável(is) Z-Score                 0  0.466667  
                          IQR                     0  0.157303  
         ≥ 2 variável(is) Z-Score                 5  0.562500  
                          IQR                     1  0.295455